# Gwen-TTS — thử voice cloning tiếng Việt trên Google Colab

Notebook này chạy model [`g-group-ai-lab/gwen-tts-0.6B`](https://huggingface.co/g-group-ai-lab/gwen-tts-0.6B), bản Qwen3-TTS 0.6B được fine-tune cho tiếng Việt.

Trước khi chạy: vào **Runtime → Change runtime type → T4 GPU** (hoặc GPU bất kỳ). Chỉ dùng audio tham chiếu khi bạn có quyền sử dụng giọng nói đó.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'Hãy chuyển runtime sang GPU rồi chạy lại cell này.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
# Qwen-TTS là runtime chính mà Gwen-TTS sử dụng.
# Không ép phiên bản torch để giữ CUDA/PyTorch tương thích với runtime Colab hiện tại.
!pip -q install -U 'qwen-tts>=0.1.1' soundfile

# FlashAttention nhanh hơn, nhưng không bắt buộc. Nếu wheel không khớp với Colab,
# notebook sẽ tự dùng PyTorch SDPA ở các cell sau.
!pip -q install flash-attn --no-build-isolation || true

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
assert uploaded, 'Hãy upload một file WAV/MP3/M4A có một người nói.'
ref_audio = next(iter(uploaded))
print('Reference audio:', ref_audio)

# Gợi ý: mẫu sạch 5–15 giây, một người nói, ít nhạc/nhiễu/reverb.

In [ ]:
# QUAN TRỌNG: transcript phải khớp với nội dung thực sự của ref_audio.
# Gwen-TTS dùng nó để căn chỉnh và clone giọng tốt hơn.
ref_text = 'Thay câu này bằng transcript CHÍNH XÁC của file audio bạn vừa upload.'
text_to_speak = (
    'Xin chào! Đây là bản thử nghiệm Gwen-TTS, mô hình chuyển văn bản thành giọng nói '
    'được tối ưu cho tiếng Việt.'
)

assert 'Thay câu này' not in ref_text, 'Hãy điền transcript cho ref_text trước khi chạy.'

In [ ]:
import importlib.util
import torch
from qwen_tts import Qwen3TTSModel

MODEL_ID = 'g-group-ai-lab/gwen-tts-0.6B'
attn_implementation = 'flash_attention_2' if importlib.util.find_spec('flash_attn') else 'sdpa'
print(f'Loading {MODEL_ID} with {attn_implementation}...')

model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map='cuda:0',
    dtype=torch.bfloat16,
    attn_implementation=attn_implementation,
)

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

# Cấu hình được tác giả Gwen-TTS khuyến nghị cho voice cloning tiếng Việt.
generation_config = {
    'temperature': 0.3,
    'top_k': 20,
    'top_p': 0.9,
    'max_new_tokens': 4096,
    'repetition_penalty': 2.0,
    'subtalker_do_sample': True,
    'subtalker_temperature': 0.1,
    'subtalker_top_k': 20,
    'subtalker_top_p': 1.0,
}

wavs, sample_rate = model.generate_voice_clone(
    text=text_to_speak,
    language='Vietnamese',
    ref_audio=ref_audio,
    ref_text=ref_text,
    **generation_config,
)

output_path = 'gwen_tts_output.wav'
sf.write(output_path, wavs[0], sample_rate)
print(f'Đã lưu {output_path} ({sample_rate} Hz)')
display(Audio(wavs[0], rate=sample_rate))

In [ ]:
from google.colab import files
files.download('gwen_tts_output.wav')

## Lưu ý khi đánh giá

- So sánh các model bằng cùng một file tham chiếu và transcript chính xác.
- Viết số, tiền tệ, ngày tháng và từ viết tắt thành cách đọc đầy đủ trước khi đưa vào TTS.
- Với đoạn dài, hãy tách thành các câu/đoạn ngắn để ổn định nhịp và ngữ điệu.
- Model checkpoint của Gwen-TTS là MIT; vẫn cần bảo đảm quyền sử dụng audio tham chiếu và tuân thủ pháp luật/phạm vi sử dụng của bạn.